# 03 — Join & Clean Data
Joins NBA salary data with player stats, applies quality filters, and exports results.

## Setup
Import required libraries and configure filter thresholds. `MIN_SALARY_CAP_PCT` excludes players on minimum contracts, and `MIN_TOTAL_MINUTES` excludes players with very limited playing time. File paths and directory names are centralized in `nba_project_utils.py` and shared across notebooks — make sure `nba_project_utils.py` is placed in the current working directory before running.

In [1]:
from __future__ import annotations
import pandas as pd

from nba_project_utils import (
    JOINED_FILE,
    SALARY_FILE,
    STATS_FILE,
    UNMATCHED_SALARY_FILE,
    UNMATCHED_STATS_FILE,
)

# Minimum salary cap percentage to include a player-season in the analysis sample
MIN_SALARY_CAP_PCT = 1.0

# Minimum total minutes played to include a player-season in the analysis sample
MIN_TOTAL_MINUTES  = 500

## Load data
Read salary and stats CSVs. Drop columns from stats that are redundant before merging.

In [2]:
# load salary and stats data from CSV
salary = pd.read_csv(SALARY_FILE)
stats  = pd.read_csv(STATS_FILE)

# Drop redundant columns from stats before merging
stats = stats.drop(columns=['SEASON', 'PLAYER_NAME', 'TEAM_ABBREVIATION'], errors='ignore')

## Join & clean
Merge on shared `id` column. Lowercase columns, drop duplicates, rename suffixed season column.

In [3]:
# join salary and stats on the shared 'id' feature using an inner join to retain only matched rows for analysis
joined = salary.merge(stats, on='id', how='inner')

# Lowercase all column names for consistency
joined.columns = joined.columns.str.lower()

# Drop redundant columns brought in from the stats dataset
joined = joined.drop(columns=['player_name', 'team_abbreviation', 'season_stats'], errors='ignore')

# Rename suffixed season column for clarity
joined = joined.rename(columns={'season_salary': 'season'})

## Compute analysis sample
Flag rows meeting minimum salary cap percentage and total minutes thresholds.

In [4]:
# compute total minutes and flag rows that meet the analysis sample criteria
joined["total_minutes"] = joined["gp"] * joined["min"]
joined["analysis_sample"] = (
    joined["salary_cap_pct"].ge(MIN_SALARY_CAP_PCT)
    & joined["total_minutes"].ge(MIN_TOTAL_MINUTES)
)

print(f"Joined rows         : {len(joined):,}")
print(f"Analysis sample rows: {int(joined['analysis_sample'].sum()):,}")

Joined rows         : 3,268
Analysis sample rows: 2,973


## Identify unmatched rows
Rows from each source dataset that did not find a match in the join — kept for diagnostics.

In [5]:
# identify unmatched rows from each dataset for diagnostics
unmatched_salary = salary[~salary['id'].isin(joined['id'])]
unmatched_stats  = stats[~stats['id'].isin(joined['id'])]

print(f"Unmatched salary rows: {len(unmatched_salary):,}")
print(f"Unmatched stats rows : {len(unmatched_stats):,}")

Unmatched salary rows: 1,152
Unmatched stats rows : 1,182


## Export CSVs
Write joined data and unmatched rows to disk.

In [6]:
# export joined data and unmatched rows to CSV
joined.to_csv(JOINED_FILE, index=False)
unmatched_salary.to_csv(UNMATCHED_SALARY_FILE, index=False)
unmatched_stats.to_csv(UNMATCHED_STATS_FILE, index=False)

print(f"Saved joined analysis data: {JOINED_FILE.name} ({len(joined):,} rows)")
print(f"Saved unmatched salary rows: {UNMATCHED_SALARY_FILE.name} ({len(unmatched_salary):,} rows)")
print(f"Saved unmatched stats rows: {UNMATCHED_STATS_FILE.name} ({len(unmatched_stats):,} rows)")
print(f"Analysis sample rows: {int(joined['analysis_sample'].sum()):,}")

Saved joined analysis data: nba_joined_analysis.csv (3,268 rows)
Saved unmatched salary rows: join_unmatched_salary_rows.csv (1,152 rows)
Saved unmatched stats rows: join_unmatched_stats_rows.csv (1,182 rows)
Analysis sample rows: 2,973
